# 🚀 Generador de Imágenes Mundos Simulados - Google Colab

Este notebook genera todas las imágenes desde los prompts del Excel y las sube a WordPress automáticamente.

**Tiempo estimado:** 3-6 horas para 720 imágenes

**Instrucciones:**
1. Sube tu Excel a Google Drive o Colab
2. Ejecuta cada celda en orden
3. Listo - las imágenes se generan y suben solas

## 1. Instalar Dependencias

In [ ]:
!pip install -q diffusers transformers torch pandas openpyxl python-dotenv requests pillow

## 2. Montar Google Drive (Opcional - para acceder a archivos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado")

## 3. Cargar Excel (desde URL o Drive)

In [ ]:
import pandas as pd
import os
from pathlib import Path

# OPCIÓN A: Si compartiste el Excel en Drive
# excel_path = '/content/drive/My Drive/Estructura Mundos Simulados.xlsx'

# OPCIÓN B: Subir manualmente con upload
# from google.colab import files
# uploaded = files.upload()
# excel_path = list(uploaded.keys())[0]

# OPCIÓN C: Descargar desde URL directa (si lo tienes en Drive/Dropbox)
# import gdown
# file_id = 'YOUR_FILE_ID_HERE'  # Del link compartido de Google Drive
# gdown.download(f'https://drive.google.com/uc?id={file_id}', 'Estructura.xlsx')
# excel_path = 'Estructura.xlsx'

# Cargar Excel
df = pd.read_excel(excel_path)
print(f"✅ Cargado Excel con {len(df)} filas")
print(f"\nColumnas: {list(df.columns)}")

## 4. Configurar Credenciales WordPress

In [ ]:
# Configuración de WordPress
WP_URL = "https://mundossimulados.online"
WP_USER = "jxaviercabellos@gmail.com"  # Tu usuario
WP_PASSWORD = "4soz XqeV beJE 9fmj wvCf ZrqI"  # Tu app password

# Codificar credenciales
import base64
credentials = base64.b64encode(f"{WP_USER}:{WP_PASSWORD}".encode()).decode()

print("✅ Credenciales de WordPress configuradas")

## 5. Configurar Modelo de Generación de Imágenes

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import io

# Usar modelo ligero para Colab
MODEL_ID = "runwayml/stable-diffusion-v1-5"

print(f"🔄 Descargando modelo: {MODEL_ID}")
print("   Esto puede tomar 2-3 minutos la primera vez...\n")

# Cargar pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None  # Desactivar para velocidad
)

pipe = pipe.to("cuda")
pipe.enable_attention_slicing()  # Optimización de memoria

print("✅ Modelo cargado y listo")

## 6. Función para Generar Imagen

In [ ]:
def generar_imagen(prompt, nombre_archivo, seed=42):
    """
    Genera una imagen usando el prompt
    Retorna: bytes de la imagen PNG
    """
    try:
        # Generar imagen
        with torch.no_grad():
            image = pipe(
                prompt,
                height=768,
                width=1024,
                num_inference_steps=30,  # Menos pasos = más rápido
                guidance_scale=7.5
            ).images[0]
        
        # Guardar en memoria como bytes
        img_bytes = io.BytesIO()
        image.save(img_bytes, format='PNG')
        img_bytes.seek(0)
        
        return img_bytes, image
    except Exception as e:
        print(f"   ❌ Error generando: {e}")
        return None, None

print("✅ Función de generación lista")

## 7. Función para Subir a WordPress

In [ ]:
import requests

def subir_a_wordpress(imagen_bytes, nombre_archivo, alt_text):
    """
    Sube imagen a WordPress Media Library
    Retorna: URL de la imagen o None si falla
    """
    try:
        # Header de autenticación
        headers = {
            'Authorization': f'Basic {credentials}'
        }
        
        # Datos de archivo
        files = {
            'file': (f"{nombre_archivo}.png", imagen_bytes, 'image/png')
        }
        
        data = {
            'title': nombre_archivo,
            'alt_text': alt_text,
            'description': alt_text
        }
        
        # Subir
        url = f"{WP_URL}/wp-json/wp/v2/media"
        response = requests.post(url, headers=headers, files=files, data=data, timeout=30)
        
        if response.status_code == 201:
            media_url = response.json().get('source_url')
            return media_url
        else:
            print(f"   ⚠️ WordPress respondió {response.status_code}")
            return None
    except Exception as e:
        print(f"   ❌ Error subiendo: {e}")
        return None

print("✅ Función de subida lista")

## 8. GENERAR Y SUBIR TODAS LAS IMÁGENES

In [ ]:
import time
from datetime import datetime

# Columnas de imágenes y prompts
columnas_imagenes = [
    ('Archivo Destacada', 'Prompt Destacada', 'Alt Text Destacada'),
    ('Archivo Interna 1', 'Prompt Interna 1', 'Alt Text Interna 1'),
    ('Archivo Interna 2', 'Prompt Interna 2', 'Alt Text Interna 2'),
]

# Registro de generación
log_generacion = []
total_imagenes = 0
total_exitosas = 0

inicio = time.time()

# Procesar cada artículo
for idx, row in df.iterrows():
    tema = row.get('Tema General (H1)', f'Artículo {idx+1}')
    print(f"\n📄 [{idx+1}/{len(df)}] {tema[:60]}...")
    
    for col_archivo, col_prompt, col_alt in columnas_imagenes:
        archivo = str(row[col_archivo]).strip()
        prompt = str(row[col_prompt]).strip() if pd.notna(row[col_prompt]) else ""
        alt_text = str(row[col_alt]).strip() if pd.notna(row[col_alt]) else archivo
        
        if not prompt:
            print(f"   ⚠️ {col_archivo}: Sin prompt")
            continue
        
        total_imagenes += 1
        
        # Generar imagen
        print(f"   🎨 {archivo}...", end="", flush=True)
        img_bytes, img = generar_imagen(prompt, archivo)
        
        if img_bytes:
            # Subir a WordPress
            print(" → 📤 Subiendo...", end="", flush=True)
            url_wp = subir_a_wordpress(img_bytes, archivo, alt_text)
            
            if url_wp:
                df.at[idx, col_archivo] = url_wp  # Guardar URL
                print(f" ✅")
                total_exitosas += 1
                log_generacion.append({
                    'archivo': archivo,
                    'estado': 'Generada y subida',
                    'url': url_wp,
                    'timestamp': datetime.now()
                })
            else:
                print(f" ⚠️ No se subió a WP")
        else:
            print(f" ❌")
    
    # Mostrar progreso
    tiempo_pasado = time.time() - inicio
    print(f"   ⏱️ {int(tiempo_pasado)}s | {total_exitosas}/{total_imagenes} exitosas")

print(f"\n{'='*70}")
print(f"✅ GENERACIÓN COMPLETADA")
print(f"   Total: {total_imagenes} | Exitosas: {total_exitosas}")
print(f"   Tiempo: {int(time.time() - inicio)}s")
print(f"{'='*70}")

## 9. Guardar Excel Actualizado

In [ ]:
# Guardar Excel actualizado
output_path = 'Estructura_Mundos_Simulados_CON_IMAGENES.xlsx'
df.to_excel(output_path, index=False)
print(f"✅ Excel guardado: {output_path}")

# Descargar
from google.colab import files
files.download(output_path)
print("📥 Descargado a tu PC")

## 10. Resumen de Generación

In [ ]:
import pandas as pd

# Crear DataFrame del log
log_df = pd.DataFrame(log_generacion)

print("\n📊 RESUMEN FINAL")
print("="*70)
print(f"Total de imágenes generadas: {total_exitosas}")
print(f"Tasa de éxito: {(total_exitosas/total_imagenes*100):.1f}%")
print(f"Tiempo total: {int(time.time() - inicio)} segundos (~{int((time.time() - inicio)/60)} minutos)")
print(f"Tiempo promedio por imagen: {((time.time() - inicio)/total_exitosas):.1f}s")
print("="*70)

if len(log_df) > 0:
    print("\nÚltimas 5 imágenes procesadas:")
    print(log_df.tail().to_string())